# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.metrics import precision_score, recall_score, make_scorer
from sklearn.linear_model import LogisticRegression
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [43]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [44]:
dayofweek = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = dayofweek.dayofweek
del dayofweek

In [45]:
X = df.drop(columns=['dayofweek'])
y = df.dayofweek

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=21, test_size=0.2, stratify=y)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [47]:
def print_scores_on_test_set(y_pred):
    print(f'accuracy is {accuracy_score(y_test, y_pred):.5f}')
    print(f'precision is {precision_score(y_test, y_pred, average='weighted'):.5f}')
    print(f'recall is {recall_score(y_test, y_pred, average='weighted'):.5f}')

In [48]:
def print_scores(cv_results):
    print(f'accuracy is {cv_results['test_accuracy'].mean():.5f}')
    print(f'precision is {cv_results['test_precision'].mean():.5f}')
    print(f'recall is {cv_results['test_recall'].mean():.5f}')

In [49]:
scoring = {
    'accuracy' : 'accuracy',
    'precision' : make_scorer(precision_score, average='weighted'),
    'recall' : make_scorer(recall_score, average='weighted' )
}

In [50]:
svc_model = SVC(random_state=21, C=10, gamma='auto', kernel='rbf', probability=True)
svc_cv_results = cross_validate(svc_model, X_train, y_train, scoring=scoring, cv=5)
print_scores(svc_cv_results)

accuracy is 0.87611
precision is 0.87904
recall is 0.87611


In [51]:
tree_model = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
tree_cv_results = cross_validate(tree_model, X_train, y_train, scoring=scoring, cv=5)
print_scores(tree_cv_results)

accuracy is 0.87312
precision is 0.87710
recall is 0.87312


In [52]:
forest_model = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)
forest_cv_results = cross_validate(forest_model, X_train, y_train, scoring=scoring, cv=5)
print_scores(forest_cv_results)

accuracy is 0.89837
precision is 0.90282
recall is 0.89837


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [53]:
vot_class_ensamble = VotingClassifier(estimators=[('svc', svc_model),
                                                  ('tree', tree_model),
                                                    ('forest', forest_model)], voting='hard',n_jobs=-1)
vot_class_cv_results = cross_validate(vot_class_ensamble, X_train, y_train, scoring=scoring, cv=5)
print_scores(vot_class_cv_results)

accuracy is 0.89614
precision is 0.90017
recall is 0.89614


In [54]:
param_grid = {
    'voting' : ['hard', 'soft'],
    'weights': [
        (x, y, z) for x in range(1,6)
                for y in range(1,6)
                for z in range(1,6)
    ]
}
vot_class_ensamble = VotingClassifier(estimators=[('svc', svc_model),
                                                  ('tree', tree_model),
                                                    ('forest', forest_model)],n_jobs=-1)

In [55]:
grid = GridSearchCV(
    estimator=vot_class_ensamble,
    param_grid=param_grid,
    cv=5,
    scoring=scoring,
    n_jobs=-1,
    refit='accuracy',
)
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=VotingClassifier(estimators=[('svc',
                                                     SVC(C=10, gamma='auto',
                                                         probability=True,
                                                         random_state=21)),
                                                    ('tree',
                                                     DecisionTreeClassifier(class_weight='balanced',
                                                                            max_depth=21,
                                                                            random_state=21)),
                                                    ('forest',
                                                     RandomForestClassifier(class_weight='balanced',
                                                                            criterion='entropy',
                                                                            max_depth=24,
                                                                            random_state=21))],
                                        n_jobs=-1),
             n_jobs=-1,
             param_grid={'...
                                     (1, 3, 3), (1, 3, 4), (1, 3, 5), (1, 4, 1),
                                     (1, 4, 2), (1, 4, 3), (1, 4, 4), (1, 4, 5),
                                     (1, 5, 1), (1, 5, 2), (1, 5, 3), (1, 5, 4),
                                     (1, 5, 5), (2, 1, 1), (2, 1, 2), (2, 1, 3),
                                     (2, 1, 4), (2, 1, 5), ...]},
             refit='accuracy',
             scoring={'accuracy': 'accuracy',
                      'precision': make_scorer(precision_score, response_method='predict', average=weighted),
                      'recall': make_scorer(recall_score, response_method='predict', average=weighted)})

In [56]:
pd.DataFrame(grid.cv_results_).sort_values(['rank_test_accuracy'])

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_voting,param_weights,params,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,...,std_test_precision,rank_test_precision,split0_test_recall,split1_test_recall,split2_test_recall,split3_test_recall,split4_test_recall,mean_test_recall,std_test_recall,rank_test_recall
203,1.329435,0.124559,0.093804,0.015527,soft,"(4, 1, 4)","{'voting': 'soft', 'weights': (4, 1, 4)}",0.925926,0.888889,0.911111,...,0.015010,1,0.925926,0.888889,0.911111,0.914498,0.881041,0.904293,0.016713,1
183,1.160751,0.081457,0.103443,0.017386,soft,"(3, 2, 4)","{'voting': 'soft', 'weights': (3, 2, 4)}",0.929630,0.892593,0.907407,...,0.015910,5,0.929630,0.892593,0.907407,0.910781,0.881041,0.904290,0.016561,2
204,1.344073,0.060026,0.081432,0.009865,soft,"(4, 1, 5)","{'voting': 'soft', 'weights': (4, 1, 5)}",0.925926,0.892593,0.903704,...,0.012998,4,0.925926,0.892593,0.903704,0.910781,0.884758,0.903552,0.014326,3
129,1.343350,0.082331,0.074531,0.008547,soft,"(1, 1, 5)","{'voting': 'soft', 'weights': (1, 1, 5)}",0.925926,0.896296,0.907407,...,0.016515,13,0.925926,0.896296,0.907407,0.910781,0.877323,0.903547,0.016176,4
233,1.283434,0.096685,0.105028,0.028549,soft,"(5, 2, 4)","{'voting': 'soft', 'weights': (5, 2, 4)}",0.933333,0.885185,0.911111,...,0.015867,2,0.933333,0.885185,0.911111,0.903346,0.884758,0.903547,0.018081,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40,1.365934,0.116033,0.076426,0.003464,hard,"(2, 4, 1)","{'voting': 'hard', 'weights': (2, 4, 1)}",0.888889,0.859259,0.903704,...,0.025028,230,0.888889,0.859259,0.903704,0.884758,0.828996,0.873121,0.026300,230
220,1.282331,0.091024,0.097206,0.016965,soft,"(4, 5, 1)","{'voting': 'soft', 'weights': (4, 5, 1)}",0.888889,0.859259,0.903704,...,0.025064,248,0.888889,0.859259,0.903704,0.884758,0.828996,0.873121,0.026300,230
166,1.257504,0.051883,0.086527,0.011660,soft,"(2, 4, 2)","{'voting': 'soft', 'weights': (2, 4, 2)}",0.888889,0.859259,0.903704,...,0.025108,240,0.888889,0.859259,0.903704,0.884758,0.828996,0.873121,0.026300,230
15,1.304410,0.134399,0.085143,0.006609,hard,"(1, 4, 1)","{'voting': 'hard', 'weights': (1, 4, 1)}",0.888889,0.859259,0.903704,...,0.025028,230,0.888889,0.859259,0.903704,0.884758,0.828996,0.873121,0.026300,230


In [57]:
grid.best_params_

{'voting': 'soft', 'weights': (4, 1, 4)}

In [58]:
best_vot_class_ensamble = VotingClassifier(estimators=[('svc', svc_model),
                                                  ('tree', tree_model),
                                                    ('forest', forest_model)],n_jobs=-1,
                                                    voting='soft', weights=[4,1,4])
best_vot_class_cv_results = cross_validate(best_vot_class_ensamble, X_train, y_train, scoring=scoring, cv=5)
print('on validation set')
print_scores(best_vot_class_cv_results)

on validation set
accuracy is 0.90429
precision is 0.90942
recall is 0.90429


In [59]:
best_vot_class_ensamble.fit(X_train, y_train)

print('Voiting classifier on test set')
print_scores_on_test_set(best_vot_class_ensamble.predict(X_test))

Voiting classifier on test set
accuracy is 0.92308
precision is 0.92610
recall is 0.92308


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [60]:
bagging_clf = BaggingClassifier(estimator=svc_model, random_state=21, n_jobs=-1)
bagging_clf_cv_results = cross_validate(bagging_clf, X_train, y_train, scoring=scoring, cv=5)
print_scores(bagging_clf_cv_results)

accuracy is 0.87834
precision is 0.88273
recall is 0.87834


In [61]:
bagging_param_grid = {
    'n_estimators' : [10,20,30,40,50,60,70,80]
}

In [62]:
bagging_grid = GridSearchCV(
    estimator=bagging_clf,
    param_grid=bagging_param_grid,
    cv=5,
    scoring=scoring,
    n_jobs=-1,
    refit='accuracy',
)
bagging_grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=BaggingClassifier(estimator=SVC(C=10, gamma='auto',
                                                       probability=True,
                                                       random_state=21),
                                         n_jobs=-1, random_state=21),
             n_jobs=-1,
             param_grid={'n_estimators': [10, 20, 30, 40, 50, 60, 70, 80]},
             refit='accuracy',
             scoring={'accuracy': 'accuracy',
                      'precision': make_scorer(precision_score, response_method='predict', average=weighted),
                      'recall': make_scorer(recall_score, response_method='predict', average=weighted)})

In [63]:
pd.DataFrame(bagging_grid.cv_results_).sort_values('rank_test_accuracy')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,params,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,...,std_test_precision,rank_test_precision,split0_test_recall,split1_test_recall,split2_test_recall,split3_test_recall,split4_test_recall,mean_test_recall,std_test_recall,rank_test_recall
3,9.356077,1.856602,5.288776,2.493674,40,{'n_estimators': 40},0.896296,0.862963,0.877778,0.884758,...,0.009578,1,0.896296,0.862963,0.877778,0.884758,0.877323,0.879824,0.010867,1
2,5.931395,1.537615,3.505186,0.853551,30,{'n_estimators': 30},0.892593,0.859259,0.885185,0.884758,...,0.010538,2,0.892593,0.859259,0.885185,0.884758,0.873606,0.879080,0.011618,2
4,7.989364,3.326632,1.802349,0.772644,50,{'n_estimators': 50},0.888889,0.862963,0.877778,0.884758,...,0.007658,4,0.888889,0.862963,0.877778,0.884758,0.877323,0.878342,0.008835,3
5,11.315975,3.263549,5.330008,2.787877,60,{'n_estimators': 60},0.896296,0.862963,0.874074,0.888476,...,0.010744,3,0.896296,0.862963,0.874074,0.888476,0.869888,0.878340,0.012258,4
1,3.872630,0.784268,2.063215,0.818653,20,{'n_estimators': 20},0.892593,0.862963,0.881481,0.881041,...,0.010637,5,0.892593,0.862963,0.881481,0.881041,0.866171,0.876850,0.010897,5
0,2.503142,0.314459,1.497139,0.144859,10,{'n_estimators': 10},0.896296,0.859259,0.874074,0.888476,...,0.012957,6,0.896296,0.859259,0.874074,0.888476,0.862454,0.876112,0.014387,6
7,13.817994,2.102903,0.775249,0.380719,80,{'n_estimators': 80},0.888889,0.862963,0.874074,0.884758,...,0.009615,7,0.888889,0.862963,0.874074,0.884758,0.862454,0.874628,0.010868,7
6,9.881689,3.018752,4.074775,1.609616,70,{'n_estimators': 70},0.888889,0.862963,0.870370,0.884758,...,0.009675,8,0.888889,0.862963,0.870370,0.884758,0.862454,0.873887,0.011006,8


In [64]:
bagging_grid.best_params_

{'n_estimators': 40}

In [65]:
best_bagging_clf = BaggingClassifier(estimator=svc_model, random_state=21, n_jobs=-1, n_estimators=10)
best_bagging_clf.fit(X_train, y_train)
print('Bagging clussifier in test set')
print_scores_on_test_set(best_bagging_clf.predict(X_test))

Bagging clussifier in test set
accuracy is 0.88757
precision is 0.89182
recall is 0.88757


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [66]:

for n in [2,3,4,5,6,7]:
    cv = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)
    stack_clf = StackingClassifier(estimators=[('svc', svc_model),
                                                ('tree', tree_model),
                                                ('forest', forest_model)],
                                                cv=cv,
                                                n_jobs=-1,
                                                passthrough=True,
                                                final_estimator=LogisticRegression(solver='liblinear'))
    
    scores = cross_validate(stack_clf, X_train, y_train, cv=5, scoring=scoring)
    
    print(f'n_splits = {n}')
    print_scores(scores)
    

n_splits = 2
accuracy is 0.89763
precision is 0.90107
recall is 0.89763
n_splits = 3
accuracy is 0.90505
precision is 0.90903
recall is 0.90505
n_splits = 4
accuracy is 0.90503
precision is 0.90788
recall is 0.90503
n_splits = 5
accuracy is 0.90282
precision is 0.90648
recall is 0.90282
n_splits = 6
accuracy is 0.90356
precision is 0.90671
recall is 0.90356
n_splits = 7
accuracy is 0.90653
precision is 0.90998
recall is 0.90653


## все результаты примерно одиннаковые, но немнго лучше по accuracy n_splits = 7

In [67]:
best_cv = StratifiedKFold(n_splits=7, random_state=21, shuffle=True)
best_stack_clf = StackingClassifier(estimators=[('svc', svc_model),
                                                ('tree', tree_model),
                                                ('forest', forest_model)],
                                                cv=best_cv,
                                                n_jobs=-1,
                                                passthrough=True,
                                                final_estimator=LogisticRegression(solver='liblinear'))
best_stack_clf.fit(X_train, y_train)
print('Stacking Classifier on test set')
print_scores_on_test_set(best_stack_clf.predict(X_test))

Stacking Classifier on test set
accuracy is 0.92899
precision is 0.93022
recall is 0.92899


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

best model is voiting classifier

In [68]:
print('Voiting Classifier on validation set')
print_scores(best_vot_class_cv_results)

Voiting Classifier on validation set
accuracy is 0.90429
precision is 0.90942
recall is 0.90429


In [69]:
print('Voiting Classifier in test set')
print_scores_on_test_set(best_vot_class_ensamble.predict(X_test))

Voiting Classifier in test set
accuracy is 0.92308
precision is 0.92610
recall is 0.92308


2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.

In [70]:
df['predict'] = best_vot_class_ensamble.predict(X)
df['is_mistake'] = (df['dayofweek'] != df['predict'])
df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek,predict,is_mistake
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,4.0,False
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,4.0,False
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,4.0,False
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,4.0,False
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,4.0,False


In [71]:
error = df[df.is_mistake]
error

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek,predict,is_mistake
38,6,18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,5.0,6.0,True
43,7,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,5.0,6.0,True
46,8,21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,5.0,6.0,True
54,5,9,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,5.0,True
69,11,21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,6.0,3.0,True
74,7,15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,True
98,8,22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,5.0,True
101,1,13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0,5.0,True
117,2,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,True
126,1,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,5.0,True


## Analyze dayofweek error

In [72]:
analize_day_of_week = pd.DataFrame(df.dayofweek.value_counts(0)).reset_index().sort_values('dayofweek')
analize_day_of_week

,dayofweek,count
5,0.0,136
2,1.0,274
4,2.0,149
0,3.0,396
6,4.0,104
3,5.0,271
1,6.0,356


In [73]:
analize_day_of_week['error_count'] = error.dayofweek.value_counts()[analize_day_of_week.dayofweek]
analize_day_of_week['percentage_error'] = (analize_day_of_week.error_count / analize_day_of_week['count']) * 100
analize_day_of_week

,dayofweek,count,error_count,percentage_error
5,0.0,136,7,5.147059
2,1.0,274,3,1.094891
4,2.0,149,2,1.342282
0,3.0,396,8,2.020202
6,4.0,104,5,4.807692
3,5.0,271,7,2.583026
1,6.0,356,6,1.685393


## analyze labname

In [74]:
lab_list = [i for i in df.columns if i.startswith('labname')]
lab_list

['labname_code_rvw',
 'labname_lab02',
 'labname_lab03',
 'labname_lab03s',
 'labname_lab05s',
 'labname_laba04',
 'labname_laba04s',
 'labname_laba05',
 'labname_laba06',
 'labname_laba06s',
 'labname_project1']

In [75]:
data = []

for lab in lab_list:
    d = {
        'labname' : lab,
        'count_lab' : df[lab].sum()
    }
    data.append(d)

analyze_labname = pd.DataFrame(data)
del data

analyze_labname

,labname,count_lab
0,labname_code_rvw,82.0
1,labname_lab02,2.0
2,labname_lab03,1.0
3,labname_lab03s,1.0
4,labname_lab05s,36.0
5,labname_laba04,178.0
6,labname_laba04s,104.0
7,labname_laba05,222.0
8,labname_laba06,48.0
9,labname_laba06s,61.0


In [76]:
data = []

for lab in lab_list:
    d = {
        'labname' : lab,
        'error_count' : error[lab].sum()
    }
    data.append(d)

analyze_labname = analyze_labname.merge(pd.DataFrame(data), on='labname')
del data

analyze_labname

,labname,count_lab,error_count
0,labname_code_rvw,82.0,4.0
1,labname_lab02,2.0,0.0
2,labname_lab03,1.0,1.0
3,labname_lab03s,1.0,0.0
4,labname_lab05s,36.0,3.0
5,labname_laba04,178.0,8.0
6,labname_laba04s,104.0,5.0
7,labname_laba05,222.0,1.0
8,labname_laba06,48.0,2.0
9,labname_laba06s,61.0,1.0


In [77]:
analyze_labname['percentage_error']  = (analyze_labname.error_count / analyze_labname.count_lab) * 100
analyze_labname

,labname,count_lab,error_count,percentage_error
0,labname_code_rvw,82.0,4.0,4.878049
1,labname_lab02,2.0,0.0,0.000000
2,labname_lab03,1.0,1.0,100.000000
3,labname_lab03s,1.0,0.0,0.000000
4,labname_lab05s,36.0,3.0,8.333333
5,labname_laba04,178.0,8.0,4.494382
6,labname_laba04s,104.0,5.0,4.807692
7,labname_laba05,222.0,1.0,0.450450
8,labname_laba06,48.0,2.0,4.166667
9,labname_laba06s,61.0,1.0,1.639344


## Analyze uid

In [78]:
uid_list = [i for i in df.columns if i.startswith('uid')]
uid_list

['uid_user_0',
 'uid_user_1',
 'uid_user_10',
 'uid_user_11',
 'uid_user_12',
 'uid_user_13',
 'uid_user_14',
 'uid_user_15',
 'uid_user_16',
 'uid_user_17',
 'uid_user_18',
 'uid_user_19',
 'uid_user_2',
 'uid_user_20',
 'uid_user_21',
 'uid_user_22',
 'uid_user_23',
 'uid_user_24',
 'uid_user_25',
 'uid_user_26',
 'uid_user_27',
 'uid_user_28',
 'uid_user_29',
 'uid_user_3',
 'uid_user_30',
 'uid_user_31',
 'uid_user_4',
 'uid_user_6',
 'uid_user_7',
 'uid_user_8']

In [79]:
data = []

for uid in uid_list:
    d = {
        'uid' : uid,
        'count_uid' : df[uid].sum()
    }
    data.append(d)

analyze_uid = pd.DataFrame(data)
del data

analyze_uid

,uid,count_uid
0,uid_user_0,2.0
1,uid_user_1,46.0
2,uid_user_10,71.0
3,uid_user_11,5.0
4,uid_user_12,49.0
5,uid_user_13,60.0
6,uid_user_14,132.0
7,uid_user_15,17.0
8,uid_user_16,32.0
9,uid_user_17,34.0


In [80]:
data = []

for uid in uid_list:
    d = {
        'uid' : uid,
        'error_count' : error[uid].sum()
    }
    data.append(d)

analyze_uid = analyze_uid.merge(pd.DataFrame(data), on='uid')
del data

analyze_uid

,uid,count_uid,error_count
0,uid_user_0,2.0,0.0
1,uid_user_1,46.0,1.0
2,uid_user_10,71.0,1.0
3,uid_user_11,5.0,0.0
4,uid_user_12,49.0,0.0
5,uid_user_13,60.0,2.0
6,uid_user_14,132.0,2.0
7,uid_user_15,17.0,1.0
8,uid_user_16,32.0,1.0
9,uid_user_17,34.0,4.0


In [81]:
analyze_uid['percentage_error']  = (analyze_uid.error_count / analyze_uid.count_uid) * 100
analyze_uid

,uid,count_uid,error_count,percentage_error
0,uid_user_0,2.0,0.0,0.000000
1,uid_user_1,46.0,1.0,2.173913
2,uid_user_10,71.0,1.0,1.408451
3,uid_user_11,5.0,0.0,0.000000
4,uid_user_12,49.0,0.0,0.000000
5,uid_user_13,60.0,2.0,3.333333
6,uid_user_14,132.0,2.0,1.515152
7,uid_user_15,17.0,1.0,5.882353
8,uid_user_16,32.0,1.0,3.125000
9,uid_user_17,34.0,4.0,11.764706


## save model

In [82]:
joblib.dump(best_vot_class_ensamble, 'best_voiting_classifier.joblib')

['best_voiting_classifier.joblib']